# Download FineMath
Streams `HuggingFaceTB/finemath` (math web content from CC, quality-scored).
Saves into `data_finemath/` with `<|endoftext|>` separators. Resumable.

Configs:
- `finemath-4plus`  (default) — 9.6B tokens, ~50 GB, highest quality (score ≥ 4)
- `finemath-3plus`  — 34B tokens, ~150 GB, broader (score ≥ 3)
- `infiwebmath-3plus` — 20.5B tokens, ~90 GB
- `infiwebmath-4plus` — 8.5B tokens, ~40 GB

In [ ]:
!pip install -q datasets

In [ ]:
import os
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/synapse/datasets_pretrain/data_finemath"
os.makedirs(OUT_DIR, exist_ok=True)

CONFIG = "finemath-4plus"   # see header for alternatives
DOCS_PER_FILE = 5_000
MIN_LENGTH = 200
MAX_DOCS = 100_000_000      # sentinel; exits when stream ends
SEPARATOR = "\n<|endoftext|>\n"
PREFIX = f"finemath_{CONFIG.replace('-', '_')}_part"

# Resume detection
existing = sorted([
    f for f in os.listdir(OUT_DIR)
    if f.startswith(f"{PREFIX}_") and f.endswith(".txt")
])
if existing:
    last_num = int(existing[-1].replace(f"{PREFIX}_", "").replace(".txt", ""))
    docs_done = (last_num + 1) * DOCS_PER_FILE
    print(f"Found {len(existing)} existing files (~{docs_done:,} docs)")
    print(f"Resuming from doc {docs_done:,}...")
else:
    docs_done = 0

print(f"\nStreaming HuggingFaceTB/finemath '{CONFIG}'...")
ds = load_dataset(
    "HuggingFaceTB/finemath",
    CONFIG,
    split="train",
    streaming=True,
)

buf = []
file_idx = len(existing)
total_chars = 0
total_docs = 0
skipped = 0

for i, example in enumerate(ds):
    if i >= MAX_DOCS:
        break
    if i < docs_done:
        if i % 100_000 == 0 and i > 0:
            print(f"  Skipping... {i:,}/{docs_done:,}")
        continue

    text = example.get("text", "").strip()
    if len(text) < MIN_LENGTH:
        skipped += 1
        continue

    buf.append(text)
    total_docs += 1

    if len(buf) >= DOCS_PER_FILE:
        path = os.path.join(OUT_DIR, f"{PREFIX}_{file_idx:04d}.txt")
        content = SEPARATOR.join(buf)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        size_mb = os.path.getsize(path) / 1024 / 1024
        total_chars += len(content)
        total_gb = total_chars / 1024 / 1024 / 1024
        print(f"  {path.split('/')[-1]}: {size_mb:.0f} MB | {total_docs:,} docs | {total_gb:.1f} GB | {skipped:,} skipped")
        buf = []
        file_idx += 1

# Write remainder
if buf:
    path = os.path.join(OUT_DIR, f"{PREFIX}_{file_idx:04d}.txt")
    content = SEPARATOR.join(buf)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    total_chars += len(content)
    file_idx += 1

total_gb = total_chars / 1024 / 1024 / 1024
print(f"\nDone!")
print(f"  {total_docs:,} docs in {file_idx} files, ~{total_gb:.1f} GB, {skipped:,} skipped")
print(f"  Saved to: {OUT_DIR}")
print(f"\nNext: run tokenizer_pipeline -> pre_train")